# 2일차: PyTorch를 사용한 딥러닝 모델 개발

## 학습 목표
- PyTorch를 사용한 MLP(Multi-Layer Perceptron) 모델 구현
- GRU(Gated Recurrent Unit) 모델을 통한 시계열 분석
- 1일차 베이스라인 모델과의 성능 비교
- 딥러닝 모델의 학습 과정 시각화

---

## 1. 라이브러리 Import 및 환경 설정

In [1]:
# !pip install torch

In [ ]:
# 기본 라이브러리
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")
import os
import pickle

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

# scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# 시드 고정
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

Using device: cpu
PyTorch version: 2.8.0


## 2. 데이터 준비

### 2.1 데이터 로드 및 전처리 (1일차 코드 재사용)

In [3]:
# 데이터 로드
df = pd.read_csv("data/ai4i2020.csv")

# XGBoost 호환을 위한 컬럼명 변경 (대괄호 제거) - 1일차와 동일
column_rename_map = {
    "Air temperature [K]": "Air_temperature_K",
    "Process temperature [K]": "Process_temperature_K",
    "Rotational speed [rpm]": "Rotational_speed_rpm",
    "Torque [Nm]": "Torque_Nm",
    "Tool wear [min]": "Tool_wear_min",
}
df.rename(columns=column_rename_map, inplace=True)

# 특성 선택
features_to_use = [
    "Type",
    "Air_temperature_K",
    "Process_temperature_K",
    "Rotational_speed_rpm",
    "Torque_Nm",
    "Tool_wear_min",
]
target = "Machine failure"

X = df[features_to_use].copy()
y = df[target].copy()

# Type 인코딩
le = LabelEncoder()
X["Type_encoded"] = le.fit_transform(X["Type"])
X = X.drop("Type", axis=1)

# Train/Test 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"학습 데이터: {X_train_scaled.shape}")
print(f"테스트 데이터: {X_test_scaled.shape}")
print(f"입력 특성 수: {X_train_scaled.shape[1]}")
print(f"클래스 불균형 비율 - 정상:고장 = {(y_train==0).sum()}:{(y_train==1).sum()}")

학습 데이터: (8000, 6)
테스트 데이터: (2000, 6)
입력 특성 수: 6
클래스 불균형 비율 - 정상:고장 = 7729:271


### 2.2 PyTorch Dataset 생성

In [4]:
from torch.utils.data import Dataset
import torch

# 1. Custom Dataset 클래스 정의
class MyDataset(Dataset):
    # NumPy 배열과 Series를 그대로 받아서 저장
    def __init__(self, X_data_np, y_data_series):
        self.X_data = X_data_np
        self.y_data = y_data_series.values.reshape(-1, 1)

    # 데이터셋의 총 길이 반환
    def __len__(self):
        return len(self.X_data)

    # 요청받은 index의 데이터만 텐서로 변환하여 반환
    def __getitem__(self, index):
        x_item = torch.FloatTensor(self.X_data[index])
        y_item = torch.FloatTensor(self.y_data[index])
        return x_item, y_item

# 2. Custom Dataset으로 데이터셋 생성 (이 과정은 매우 빠릅니다)
train_dataset = MyDataset(X_train_scaled, y_train)
test_dataset = MyDataset(X_test_scaled, y_test)

# 3. DataLoader 생성 (이 부분은 동일)
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Custom Dataset 및 DataLoader 생성 완료!")
print(f"학습 배치 수: {len(train_loader)}")

Custom Dataset 및 DataLoader 생성 완료!
학습 배치 수: 125


In [5]:
train_dataset = MyDataset(X_train_scaled, y_train)
test_dataset = MyDataset(X_test_scaled, y_test)

# 3. DataLoader 생성 (이 부분은 동일)
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Custom Dataset 및 DataLoader 생성 완료!")
print(f"학습 배치 수: {len(train_loader)}")

Custom Dataset 및 DataLoader 생성 완료!
학습 배치 수: 125


In [6]:
# # NumPy 배열을 PyTorch 텐서로 변환
# X_train_tensor = torch.FloatTensor(X_train_scaled)
# y_train_tensor = torch.FloatTensor(y_train.values)
# X_test_tensor = torch.FloatTensor(X_test_scaled)
# y_test_tensor = torch.FloatTensor(y_test.values)

# # TensorDataset 생성
# train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
# test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# # DataLoader 생성
# batch_size = 64
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# print(f"배치 크기: {batch_size}")
# print(f"학습 배치 수: {len(train_loader)}")
# print(f"테스트 배치 수: {len(test_loader)}")

## 3. MLP (Multi-Layer Perceptron) 모델

### 3.1 MLP 모델 정의

In [7]:
# MLP 모델을 nn.Sequential로 직접 구현
input_size = X_train_scaled.shape[1]
hidden_sizes = [128, 64, 32]
dropout_rate = 0.3

# nn.Sequential을 사용한 MLP 모델 구성
mlp_model = nn.Sequential(
    # 첫 번째 은닉층 (입력층 -> 128)
    nn.Linear(input_size, hidden_sizes[0]),
    nn.ReLU(),
    nn.Dropout(dropout_rate),
    # 두 번째 은닉층 (128 -> 64)
    nn.Linear(hidden_sizes[0], hidden_sizes[1]),
    nn.ReLU(),
    nn.Dropout(dropout_rate),
    # 세 번째 은닉층 (64 -> 32)
    nn.Linear(hidden_sizes[1], hidden_sizes[2]),
    nn.ReLU(),
    # 출력층 (32 -> 1)
    nn.Linear(hidden_sizes[2], 1),
    nn.Sigmoid(),
).to(device)

print("MLP 모델 구조 (nn.Sequential 직접 구현):")
print(mlp_model)
print(f"\n총 파라미터 수: {sum(p.numel() for p in mlp_model.parameters()):,}")

# 각 층별 설명
print("\n=== 모델 구조 설명 ===")
print("1. Linear(6 -> 128): 입력 특성 6개를 128개 뉴런으로 변환")
print("2. ReLU: 비선형 활성화 함수")
print("3. Dropout(0.3): 과적합 방지를 위한 30% 드롭아웃")
print("4. 위 과정을 128->64, 64->32로 반복")
print("5. Linear(32 -> 1): 최종 출력 (이진 분류)")
print("6. Sigmoid: 0~1 사이 확률값으로 변환")

MLP 모델 구조 (nn.Sequential 직접 구현):
Sequential(
  (0): Linear(in_features=6, out_features=128, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=128, out_features=64, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.3, inplace=False)
  (6): Linear(in_features=64, out_features=32, bias=True)
  (7): ReLU()
  (8): Linear(in_features=32, out_features=1, bias=True)
  (9): Sigmoid()
)

총 파라미터 수: 11,265

=== 모델 구조 설명 ===
1. Linear(6 -> 128): 입력 특성 6개를 128개 뉴런으로 변환
2. ReLU: 비선형 활성화 함수
3. Dropout(0.3): 과적합 방지를 위한 30% 드롭아웃
4. 위 과정을 128->64, 64->32로 반복
5. Linear(32 -> 1): 최종 출력 (이진 분류)
6. Sigmoid: 0~1 사이 확률값으로 변환


### 3.2 학습 함수 정의

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """한 에폭 학습"""
    model.train()
    total_loss = 0
    all_predictions = []
    all_targets = []

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 통계 업데이트
        total_loss += loss.item()
        predicted = (outputs > 0.5).float()

        all_predictions.extend(predicted.cpu().numpy())
        all_targets.extend(batch_y.cpu().numpy())

    avg_loss = total_loss / len(train_loader)

    # F1 score 계산
    f1 = f1_score(all_targets, all_predictions)
    accuracy = accuracy_score(all_targets, all_predictions)

    return avg_loss, accuracy, f1


def evaluate(model, test_loader, criterion, device):
    """모델 평가"""
    model.eval()
    total_loss = 0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)

            total_loss += loss.item()
            predicted = (outputs > 0.5).float()

            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(batch_y.cpu().numpy())

    avg_loss = total_loss / len(test_loader)

    # 메트릭 계산
    accuracy = accuracy_score(all_targets, all_predictions)
    f1 = f1_score(all_targets, all_predictions)

    return avg_loss, accuracy, f1, all_predictions, all_targets

: 

### 3.3 MLP 모델 학습

In [ ]:
# 학습 설정
num_epochs = 200
learning_rate = 0.001

# 손실 함수와 옵티마이저
criterion = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = optim.Adam(mlp_model.parameters(), lr=learning_rate)

# 학습률 스케줄러
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

# 학습 기록 저장
mlp_history = {
    "train_loss": [],
    "train_acc": [],
    "train_f1": [],
    "val_loss": [],
    "val_acc": [],
    "val_f1": [],
}

print("MLP 모델 학습 시작...")
print("=" * 50)

best_val_f1 = 0  # F1 score 기준으로 최고 모델 선택
best_epoch = 0

for epoch in range(num_epochs):
    # 학습
    train_loss, train_acc, train_f1 = train_epoch(
        mlp_model, train_loader, criterion, optimizer, device
    )

    # 평가
    val_loss, val_acc, val_f1, _, _ = evaluate(
        mlp_model, test_loader, criterion, device
    )

    # 학습률 조정
    scheduler.step(val_loss)

    # 기록 저장
    mlp_history["train_loss"].append(train_loss)
    mlp_history["train_acc"].append(train_acc)
    mlp_history["train_f1"].append(train_f1)
    mlp_history["val_loss"].append(val_loss)
    mlp_history["val_acc"].append(val_acc)
    mlp_history["val_f1"].append(val_f1)

    # 최고 모델 저장 (F1 score 기준)
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch + 1
        torch.save(mlp_model.state_dict(), "best_mlp_model.pth")

    # 진행상황 출력 (10 에폭마다)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(
            f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}"
        )
        print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")

print("=" * 50)
print(f"✅ 학습 완료! 최고 검증 F1 Score: {best_val_f1:.4f} (Epoch {best_epoch})")

### 3.4 학습 곡선 시각화

In [ ]:
# 학습 곡선 그리기
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss 곡선
axes[0].plot(mlp_history["train_loss"], label="Train Loss", linewidth=2)
axes[0].plot(mlp_history["val_loss"], label="Validation Loss", linewidth=2)
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Loss", fontsize=12)
axes[0].set_title("MLP Training Loss", fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy 곡선
axes[1].plot(mlp_history["train_acc"], label="Train Accuracy", linewidth=2)
axes[1].plot(mlp_history["val_acc"], label="Validation Accuracy", linewidth=2)
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Accuracy", fontsize=12)
axes[1].set_title("MLP Training Accuracy", fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1 Score 곡선
axes[2].plot(
    mlp_history["train_f1"], label="Train F1 Score", linewidth=2, color="green"
)
axes[2].plot(
    mlp_history["val_f1"], label="Validation F1 Score", linewidth=2, color="red"
)
axes[2].set_xlabel("Epoch", fontsize=12)
axes[2].set_ylabel("F1 Score", fontsize=12)
axes[2].set_title("MLP Training F1 Score", fontsize=14)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.5 MLP 모델 최종 평가

In [ ]:
# 최고 모델 로드
mlp_model.load_state_dict(torch.load("best_mlp_model.pth"))

# 최종 평가
_, _, _, mlp_predictions, mlp_targets = evaluate(
    mlp_model, test_loader, criterion, device
)

# 성능 메트릭 계산
mlp_metrics = {
    "Accuracy": accuracy_score(mlp_targets, mlp_predictions),
    "Precision": precision_score(mlp_targets, mlp_predictions),
    "Recall": recall_score(mlp_targets, mlp_predictions),
    "F1-Score": f1_score(mlp_targets, mlp_predictions),
}

print("=== MLP 모델 최종 성능 (테스트 데이터) ===")
for metric, value in mlp_metrics.items():
    print(f"{metric:10s}: {value:.4f}")

# Confusion Matrix
plt.figure(figsize=(6, 5))
cm = confusion_matrix(mlp_targets, mlp_predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Failure"])
disp.plot(cmap="Blues")
plt.title("MLP Confusion Matrix", fontsize=14)
plt.show()

# 성능 상세 분석
print("\n=== 성능 상세 분석 ===")
print(f"True Positives (고장을 고장으로): {cm[1, 1]}")
print(f"True Negatives (정상을 정상으로): {cm[0, 0]}")
print(f"False Positives (정상을 고장으로): {cm[0, 1]}")
print(f"False Negatives (고장을 정상으로): {cm[1, 0]}")

## 4. GRU (Gated Recurrent Unit) 모델

### 4.1 시계열 데이터 생성

In [ ]:
def create_sequences(X, y, seq_length=10):
    """
    시계열 시퀀스 생성 - 슬라이딩 윈도우 방식

    Args:
        X: 입력 데이터 (8000, 6) - 8000개 샘플, 6개 특성
        y: 타겟 데이터 (8000,) - 8000개 레이블
        seq_length: 시퀀스 길이 (윈도우 크기)

    Returns:
        X_seq: 시퀀스 형태의 입력 데이터 (N, seq_length, 6)
        y_seq: 시퀀스에 대응하는 타겟 (N,)
    """
    X_seq = []
    y_seq = []

    for i in range(len(X) - seq_length + 1):
        # i번째부터 i+seq_length까지의 데이터를 하나의 시퀀스로 만듦
        X_seq.append(X[i : i + seq_length])
        # 시퀀스의 마지막 시점의 레이블을 타겟으로 사용
        y_seq.append(y[i + seq_length - 1])

    return np.array(X_seq), np.array(y_seq)


# 시퀀스 길이 설정
seq_length = 5

print("=== 시계열 데이터 변환 과정 설명 ===")
print(f"원본 데이터: {X_train_scaled.shape[0]}개 샘플")
print(f"시퀀스 길이: {seq_length}")
print(f"생성되는 시퀀스 수: {X_train_scaled.shape[0] - seq_length + 1}개")
print()
print("예시:")
print("원본 데이터가 [샘플1, 샘플2, 샘플3, 샘플4, 샘플5, 샘플6, 샘플7]이고")
print("시퀀스 길이가 5라면:")
print("  시퀀스1: [샘플1, 샘플2, 샘플3, 샘플4, 샘플5] -> 레이블: 샘플5의 레이블")
print("  시퀀스2: [샘플2, 샘플3, 샘플4, 샘플5, 샘플6] -> 레이블: 샘플6의 레이블")
print("  시퀀스3: [샘플3, 샘플4, 샘플5, 샘플6, 샘플7] -> 레이블: 샘플7의 레이블")
print()
print("이렇게 슬라이딩 윈도우 방식으로 시퀀스를 생성합니다.")
print("GRU는 이전 시점들의 정보를 종합하여 현재 시점을 예측하게 됩니다.")
print()

# 시계열 데이터 생성
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train.values, seq_length)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test.values, seq_length)

print("=== 변환 결과 ===")
print(f"학습 데이터: {X_train_scaled.shape} -> {X_train_seq.shape}")
print(f"  - 원본: {X_train_scaled.shape[0]} 샘플 × {X_train_scaled.shape[1]} 특성")
print(
    f"  - 시계열: {X_train_seq.shape[0]} 시퀀스 × {X_train_seq.shape[1]} 시간스텝 × {X_train_seq.shape[2]} 특성"
)
print(f"테스트 데이터: {X_test_scaled.shape} -> {X_test_seq.shape}")
print(f"  - 원본: {X_test_scaled.shape[0]} 샘플 × {X_test_scaled.shape[1]} 특성")
print(
    f"  - 시계열: {X_test_seq.shape[0]} 시퀀스 × {X_test_seq.shape[1]} 시간스텝 × {X_test_seq.shape[2]} 특성"
)

In [ ]:
# GRU용 DataLoader 생성
X_train_seq_tensor = torch.FloatTensor(X_train_seq)
y_train_seq_tensor = torch.FloatTensor(y_train_seq)
X_test_seq_tensor = torch.FloatTensor(X_test_seq)
y_test_seq_tensor = torch.FloatTensor(y_test_seq)

train_seq_dataset = TensorDataset(X_train_seq_tensor, y_train_seq_tensor)
test_seq_dataset = TensorDataset(X_test_seq_tensor, y_test_seq_tensor)

train_seq_loader = DataLoader(train_seq_dataset, batch_size=batch_size, shuffle=True)
test_seq_loader = DataLoader(test_seq_dataset, batch_size=batch_size, shuffle=False)

print(f"GRU 학습 배치 수: {len(train_seq_loader)}")
print(f"GRU 테스트 배치 수: {len(test_seq_loader)}")

### 4.2 GRU 모델 정의

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=2, dropout_rate=0.3):
        """
        GRU 모델 정의
        Args:
            input_size: 입력 특성 수
            hidden_size: GRU 은닉 상태 크기
            num_layers: GRU 층 수
            dropout_rate: 드롭아웃 비율
        """
        super(GRUModel, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # GRU 층
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0,
        )

        # Fully Connected 층
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # GRU forward pass
        # x shape: (batch_size, sequence_length, input_size)
        out, hidden = self.gru(x)

        # 마지막 타임스텝의 출력만 사용
        out = out[:, -1, :]  # (batch_size, hidden_size)

        # Fully Connected layers
        out = self.dropout(out)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)

        return out.squeeze()


# GRU 모델 인스턴스 생성
input_size = X_train_seq.shape[2]  # 특성 수
hidden_size = 64
num_layers = 2

gru_model = GRUModel(input_size, hidden_size, num_layers).to(device)

print("GRU 모델 구조:")
print(gru_model)
print(f"\n총 파라미터 수: {sum(p.numel() for p in gru_model.parameters()):,}")

### 4.3 GRU 모델 학습

In [ ]:
# 학습 설정
num_epochs = 200
learning_rate = 0.001

# 손실 함수와 옵티마이저
criterion = nn.BCELoss()
optimizer = optim.Adam(gru_model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

# 학습 기록 저장
gru_history = {
    "train_loss": [],
    "train_acc": [],
    "train_f1": [],
    "val_loss": [],
    "val_acc": [],
    "val_f1": [],
}

print("GRU 모델 학습 시작...")
print("=" * 50)

best_val_f1 = 0  # F1 score 기준으로 최고 모델 선택
best_epoch = 0

for epoch in range(num_epochs):
    # 학습
    train_loss, train_acc, train_f1 = train_epoch(
        gru_model, train_seq_loader, criterion, optimizer, device
    )

    # 평가
    val_loss, val_acc, val_f1, _, _ = evaluate(
        gru_model, test_seq_loader, criterion, device
    )

    # 학습률 조정
    scheduler.step(val_loss)

    # 기록 저장
    gru_history["train_loss"].append(train_loss)
    gru_history["train_acc"].append(train_acc)
    gru_history["train_f1"].append(train_f1)
    gru_history["val_loss"].append(val_loss)
    gru_history["val_acc"].append(val_acc)
    gru_history["val_f1"].append(val_f1)

    # 최고 모델 저장 (F1 score 기준)
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch + 1
        torch.save(gru_model.state_dict(), "best_gru_model.pth")

    # 진행상황 출력 (10 에폭마다)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(
            f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}"
        )
        print(f"  Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")

print("=" * 50)
print(f"✅ 학습 완료! 최고 검증 F1 Score: {best_val_f1:.4f} (Epoch {best_epoch})")

### 4.4 GRU 학습 곡선 시각화

In [ ]:
# 학습 곡선 그리기
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss 곡선
axes[0].plot(gru_history["train_loss"], label="Train Loss", linewidth=2)
axes[0].plot(gru_history["val_loss"], label="Validation Loss", linewidth=2)
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Loss", fontsize=12)
axes[0].set_title("GRU Training Loss", fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy 곡선
axes[1].plot(gru_history["train_acc"], label="Train Accuracy", linewidth=2)
axes[1].plot(gru_history["val_acc"], label="Validation Accuracy", linewidth=2)
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Accuracy", fontsize=12)
axes[1].set_title("GRU Training Accuracy", fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1 Score 곡선
axes[2].plot(
    gru_history["train_f1"], label="Train F1 Score", linewidth=2, color="green"
)
axes[2].plot(
    gru_history["val_f1"], label="Validation F1 Score", linewidth=2, color="red"
)
axes[2].set_xlabel("Epoch", fontsize=12)
axes[2].set_ylabel("F1 Score", fontsize=12)
axes[2].set_title("GRU Training F1 Score", fontsize=14)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.5 GRU 모델 최종 평가

In [ ]:
# 최고 모델 로드
gru_model.load_state_dict(torch.load("best_gru_model.pth"))

# 최종 평가
_, _, _, gru_predictions, gru_targets = evaluate(
    gru_model, test_seq_loader, criterion, device
)

# 성능 메트릭 계산
gru_metrics = {
    "Accuracy": accuracy_score(gru_targets, gru_predictions),
    "Precision": precision_score(gru_targets, gru_predictions),
    "Recall": recall_score(gru_targets, gru_predictions),
    "F1-Score": f1_score(gru_targets, gru_predictions),
}

print("=== GRU 모델 최종 성능 (테스트 데이터) ===")
for metric, value in gru_metrics.items():
    print(f"{metric:10s}: {value:.4f}")

# Confusion Matrix
plt.figure(figsize=(6, 5))
cm = confusion_matrix(gru_targets, gru_predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Failure"])
disp.plot(cmap="Greens")
plt.title("GRU Confusion Matrix", fontsize=14)
plt.show()

# 성능 상세 분석
print("\n=== 성능 상세 분석 ===")
print(f"True Positives (고장을 고장으로): {cm[1, 1]}")
print(f"True Negatives (정상을 정상으로): {cm[0, 0]}")
print(f"False Positives (정상을 고장으로): {cm[0, 1]}")
print(f"False Negatives (고장을 정상으로): {cm[1, 0]}")

## 5. 모델 성능 종합 비교

### 5.1 1일차 모델 결과 불러오기

In [ ]:
# 1일차에서 학습한 모델 불러오기
print("=== 1일차 모델 불러오기 ===")

if os.path.exists("models/rf_model.pkl") and os.path.exists("models/xgb_model.pkl"):
    # 저장된 모델 로드
    with open("models/rf_model.pkl", "rb") as f:
        rf_model = pickle.load(f)
    with open("models/xgb_model.pkl", "rb") as f:
        xgb_model = pickle.load(f)
    print("✅ 1일차 모델을 성공적으로 불러왔습니다.")

    # 테스트 데이터에 대해 예측 수행
    rf_pred = rf_model.predict(X_test_scaled)
    xgb_pred = xgb_model.predict(X_test_scaled)

    # 성능 메트릭 계산
    rf_metrics = {
        "Accuracy": accuracy_score(y_test, rf_pred),
        "Precision": precision_score(y_test, rf_pred),
        "Recall": recall_score(y_test, rf_pred),
        "F1-Score": f1_score(y_test, rf_pred),
    }

    xgb_metrics = {
        "Accuracy": accuracy_score(y_test, xgb_pred),
        "Precision": precision_score(y_test, xgb_pred),
        "Recall": recall_score(y_test, xgb_pred),
        "F1-Score": f1_score(y_test, xgb_pred),
    }

    print("\n=== 1일차 모델 성능 (테스트 데이터) ===")
    print("Random Forest:")
    for metric, value in rf_metrics.items():
        print(f"  {metric:10s}: {value:.4f}")
    print("\nXGBoost:")
    for metric, value in xgb_metrics.items():
        print(f"  {metric:10s}: {value:.4f}")

else:
    # 모델 파일이 없는 경우 예시값 사용
    print("⚠️ 1일차 모델 파일이 없습니다. 예시 값을 사용합니다.")
    print("   1일차 노트북을 실행하여 모델을 저장해주세요.")

    rf_metrics = {
        "Accuracy": 0.9725,
        "Precision": 0.9643,
        "Recall": 0.1327,
        "F1-Score": 0.2333,
    }

    xgb_metrics = {
        "Accuracy": 0.9730,
        "Precision": 1.0000,
        "Recall": 0.1327,
        "F1-Score": 0.2342,
    }

### 5.2 전체 모델 성능 비교

In [ ]:
# 모든 모델의 성능 정리
all_models_performance = pd.DataFrame(
    {
        "Random Forest": rf_metrics,
        "XGBoost": xgb_metrics,
        "MLP": mlp_metrics,
        "GRU": gru_metrics,
    }
).T

print("=== 전체 모델 성능 비교 (테스트 데이터) ===")
print(all_models_performance.round(4))

# 최고 성능 모델 찾기
best_model = all_models_performance["F1-Score"].idxmax()
best_f1 = all_models_performance["F1-Score"].max()
print(f"\n🏆 최고 F1-Score 모델: {best_model} ({best_f1:.4f})")

best_recall_model = all_models_performance["Recall"].idxmax()
best_recall = all_models_performance["Recall"].max()
print(f"🏆 최고 Recall 모델: {best_recall_model} ({best_recall:.4f})")

In [ ]:
# 성능 비교 시각화
metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
models = ["Random Forest", "XGBoost", "MLP", "GRU"]
colors = ["steelblue", "lightgreen", "coral", "gold"]

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(metrics))
width = 0.2

for i, model in enumerate(models):
    values = [all_models_performance.loc[model, metric] for metric in metrics]
    bars = ax.bar(x + i * width, values, width, label=model, color=colors[i], alpha=0.8)

    # 값 표시
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            height + 0.01,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

ax.set_xlabel("Metrics", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Comprehensive Model Performance Comparison", fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics)
ax.legend(loc="upper left")
ax.set_ylim([0, 1.15])
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### 5.3 모델별 특징 분석

In [ ]:
# 모델 특성 비교
model_characteristics = pd.DataFrame(
    {
        "Model": ["Random Forest", "XGBoost", "MLP", "GRU"],
        "Type": ["Tree-based", "Tree-based", "Deep Learning", "Deep Learning"],
        "Parameters": [
            "-",
            "-",
            f"{sum(p.numel() for p in mlp_model.parameters()):,}",
            f"{sum(p.numel() for p in gru_model.parameters()):,}",
        ],
        "Training Time": ["Fast", "Fast", "Medium", "Slow"],
        "Interpretability": ["High", "Medium", "Low", "Low"],
        "Sequential Data": ["No", "No", "No", "Yes"],
    }
)

print("=== 모델 특성 비교 ===")
print(model_characteristics.to_string(index=False))

## 6. 결론 및 인사이트

In [ ]:
print("=" * 60)
print("📊 2일차 실습 결과 요약")
print("=" * 60)

print("\n1. 모델 성능 순위 (F1-Score 기준)")
f1_ranking = all_models_performance["F1-Score"].sort_values(ascending=False)
for i, (model, score) in enumerate(f1_ranking.items(), 1):
    print(f"   {i}. {model}: {score:.4f}")

print("\n2. 딥러닝 모델 특징")
print("   - MLP: nn.Sequential로 간단하게 구현, 빠른 학습")
print("   - GRU: 시계열 패턴 학습 가능, 슬라이딩 윈도우 방식")
print("   - F1 Score 기준 최적화로 불균형 데이터 대응")

print("\n3. 주요 발견사항")
print("   - 클래스 불균형이 모든 모델 성능에 영향")
print("   - 딥러닝 모델이 Tree-based 모델보다 높은 Recall")
print("   - F1 Score 기준으로 학습시 균형잡힌 성능")

print("\n4. 개선 방향")
print("   - 클래스 가중치 조정 또는 오버샘플링")
print("   - 하이퍼파라미터 튜닝 (3일차 실습)")
print("   - 앙상블 방법 적용")
print("   - Feature Engineering 추가")

print("\n5. 실무 적용 시 고려사항")
print("   - 실시간 예측이 필요한 경우: Tree-based 모델 선호")
print("   - 높은 Recall이 중요한 경우: MLP 모델 추천")
print("   - 해석가능성이 중요한 경우: Random Forest 추천")
print("   - 시계열 패턴이 중요한 경우: GRU 모델 고려")
print("=" * 60)

## 💡 실습 과제

1. **모델 아키텍처 실험**
   - MLP: 층 수와 뉴런 수 변경
   - GRU: Multi-Layer로 변경
   - 드롭아웃 비율 조정

2. **학습 최적화**
   - 다른 옵티마이저 사용 (SGD, RMSprop)
   - 학습률 스케줄링 전략 변경
   - Early Stopping 구현
   - GRU overfitting 방지를 위한 다양한 비교 실험


3. **클래스 불균형 처리**
   - Weighted Loss 함수 사용
   - Focal Loss 구현
   - 오버샘플링/언더샘플링 적용